##1. Load, View Data

In [ ]:
  from google.colab import drive
drive.mount('/content/drive')
print("--- Kết nối Drive thành công! ---")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
sns.set(style='darkgrid')
import matplotlib.pyplot as plt

In [ ]:
# Loading the dataset
data1 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Monday-WorkingHours.pcap_ISCX.csv')
data2 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Tuesday-WorkingHours.pcap_ISCX.csv')
data3 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Wednesday-workingHours.pcap_ISCX.csv')
data4 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')
data5 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv')
data6 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Friday-WorkingHours-Morning.pcap_ISCX.csv')
data7 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
data8 = pd.read_csv('/content/drive/MyDrive/DoAn_NIDS/Dataset/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')

In [ ]:
data_list = [data1, data2, data3, data4, data5, data6, data7, data8]

print('Kích thước dữ liệu: ')
for i, data in enumerate(data_list, start = 1):
  rows, cols = data.shape
  print(f'Data{i} -> {rows} rows, {cols} columns')

Kết nối dữ liệu:

In [ ]:
data = pd.concat(data_list)
rows, cols = data.shape

print('Kích thước mới')
print(f'Số hàng: {rows}')
print(f'Số cột: {cols}')
print(f'Tổng số ô: {rows * cols}')

In [ ]:
## Xóa các khung dữ liệu sau khi nối để tiết kiệm bộ nhớ
for d in data_list: del d

In [ ]:
# 1. Làm sạch tên cột
print("Đang làm sạch tên cột...")
data.columns = data.columns.str.strip()
print("Tên cột sau khi làm sạch: ")
data.columns

In [ ]:
pd.options.display.max_columns = 80
data

##Dọn dẹp dữ liệu

###Phân tích Hàng trùng lặp (Duplicates)

In [ ]:
# Đếm số lượng hàng trùng lặp
duplicate_count = data.duplicated().sum()
total_rows = len(data)

print(f"--- Phân tích Hàng trùng lặp ---")
print(f"Tổng số hàng: {total_rows}")
print(f"Số hàng trùng lặp: {duplicate_count}")
print(f"Tỷ lệ trùng lặp: {(duplicate_count / total_rows) * 100:.2f}%")

In [ ]:
# Xóa các hàng trùng lặp
data.drop_duplicates(inplace = True)
data.shape

###Phân tích các giá trị NaN, Infinity

In [ ]:
# 1. Đếm tổng số giá trị BỊ THIẾU (NaN) trong mỗi cột
print("--- Đang kiểm tra giá trị NaN (Bị thiếu) ---")
# Sẽ chỉ hiển thị các cột CÓ NaN
nan_counts = data.isna().sum()
print(nan_counts[nan_counts > 0])

# 2. Đếm tổng số giá trị VÔ CỰC (Infinity)
# Chúng ta phải thay thế 'inf' bằng NaN để .isnull() có thể đếm được
print("\n--- Đang kiểm tra giá trị Infinity (Vô cực) ---")
# Thay thế mọi giá trị 'inf' và '-inf' bằng NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3. Đếm lại NaN (lần này sẽ bao gồm cả các giá trị 'inf' cũ)
print("\n--- Tổng NaN (bao gồm cả Infinity cũ) ---")
nan_counts_total = data.isnull().sum()
print(nan_counts_total[nan_counts_total > 0])

Phân tích Phân bố

In [ ]:

print("Đang sửa lỗi index trùng lặp...")
# drop=True: Bỏ index cũ đi, không biến nó thành một cột mới
# inplace=True: Sửa trực tiếp DataFrame 'data'
data.reset_index(drop=True, inplace=True)

print("--- Sửa lỗi Index hoàn tất! ---")

In [ ]:
# Biểu đồ 1: Boxplot cho Flow Bytes/s (Dùng Thang đo Log)
print("Đang vẽ Boxplot cho 'Flow Bytes/s' (với Thang đo Log)...")

plt.figure(figsize = (8, 4))
sns.boxplot(x = data['Flow Bytes/s'])
plt.xscale('log')  # <-- DÒNG QUAN TRỌNG NHẤT
plt.title('Boxplot của Flow Bytes/s (Thang đo Log)')
plt.xlabel('Flow Bytes/s (Log Scale)')
plt.show()

# In ra giá trị Mean và Median để so sánh
col_mean = data['Flow Bytes/s'].mean()
col_median = data['Flow Bytes/s'].median()
print(f"  - Mean (Trung bình): {col_mean:.2f}")
print(f"  - Median (Trung vị): {col_median:.2f}")

 Nhận xét về Biểu đồ (Phân tích Flow Bytes/s):

  1. **Dữ liệu Bị lệch nặng (Heavily Skewed)**: Con số Mean (Trung bình) là 1,410,706 trong khi Median (Trung vị) chỉ là 3,715. Việc Mean lớn hơn Median tới gần 380 lần là bằng chứng không thể chối cãi rằng dữ liệu bị "lệch phải" (right-skewed). Điều này có nghĩa là đa số dữ liệu có giá trị thấp, nhưng bị ảnh hưởng bởi một số ít giá trị ngoại lai (outliers) cực kỳ lớn.
    
  2. **Giá trị Ngoại lai (Outliers)**: Biểu đồ cho thấy "cái hộp" (đại diện cho 50% dữ liệu) nằm ở bên trái (khoảng $10^2$ - $10^5$). Tuy nhiên, có một lượng lớn các điểm ngoại lai (chấm đen) kéo dài từ $10^6$ đến $10^9$. Đây chính là các giá trị (thường là tấn công DDoS) đã "kéo" giá trị Mean (Trung bình) lên cao.
  
Kết luận: Phân tích này chứng minh rằng Median (Trung vị) là một giá trị đại diện "bền bỉ" (robust) và chính xác hơn Mean (Trung bình). Do đó, việc lấp đầy (fill) các giá trị NaN bằng Median (3715.04) là lựa chọn hợp lý nhất về mặt thống kê.

In [ ]:
# Biểu đồ 2: Boxplot cho Flow Packets/s (Dùng Thang đo Log)
print("Đang vẽ Boxplot cho 'Flow Packets/s' (với Thang đo Log)...")

plt.figure(figsize = (8, 4))
# Vẽ boxplot cho cột 'Flow Packets/s'
sns.boxplot(x = data['Flow Packets/s'])
plt.xscale('log')  # <-- Dùng thang đo Log
plt.title('Boxplot của Flow Packets/s (Thang đo Log)')
plt.xlabel('Flow Packets/s (Log Scale)')
plt.show()

# In ra giá trị Mean và Median để so sánh
col_mean = data['Flow Packets/s'].mean()
col_median = data['Flow Packets/s'].median()
print(f"  - Mean (Trung bình): {col_mean:.2f}")
print(f"  - Median (Trung vị): {col_median:.2f}")

 **Nhận xét về Biểu đồ (Phân tích Flow Packets/s):**
  1. Mức độ Lệch Cực kỳ Cao: Mean (Trung bình) là 47,291 trong khi Median (Trung vị) chỉ là 69.74. Mean lớn hơn Median đến hơn 670 lần. Đây là một trường hợp "lệch phải" (right-skewed) rất điển hình, cho thấy các giá trị ngoại lai (outliers) đang "phá vỡ" hoàn toàn giá trị trung bình.
  2. Hành vi Điển hình: Biểu đồ hộp (boxplot) cho thấy 50% dữ liệu "điển hình" nằm trong khoảng từ $10^1$ đến $10^4$. Giá trị Median (69.74) đại diện rất tốt cho phần lớn dữ liệu này.
  3. Giá trị Ngoại lai (Outliers): Các điểm đen (outliers) kéo dài từ $10^5$ trở đi. Đây là những luồng mạng (flows) có số lượng gói tin/giây cực cao (ví dụ: tấn công DoS/DDoS), và chúng chính là thủ phạm kéo Mean (Trung bình) lên một con số "vô nghĩa" là 47,291.
  
 **Kết luận chung:** Cả hai phân tích (Flow Bytes/s và Flow Packets/s) đều CHỨNG MINH một điều: Dữ liệu bị lệch nặng.

Lấp đầy các giá trị NaN/Infinity bằng giá trị Trung vị

In [ ]:
print("\n--- Bắt đầu DỌN DẸP CUỐI CÙNG ---")

# Lấp đầy bằng Trung vị
cols_to_fill_median = ['Flow Bytes/s', 'Flow Packets/s']
for col in cols_to_fill_median:
    median_val = data[col].median()
    data[col] = data[col].fillna(median_val)

print(f"Đã lấp đầy {cols_to_fill_median} bằng Trung vị.")

# Lấp đầy tất cả các NaN CÒN LẠI (nếu có) bằng 0
data.fillna(0, inplace=True)
print("Tất cả các NaN còn lại đã được fill bằng 0.")

# Kiểm tra lại
print(f"Dữ liệu còn NaN không? : {data.isnull().sum().any()}")
print("\n--- DỌN DẸP HOÀN TẤT! ---")

In [ ]:
# Lưu file Parquet (Cuối cùng)
print("--- Đang lưu bộ dữ liệu đã làm sạch (Nâng cao) ra file Parquet... ---")

save_file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Cleaned_Base.parquet"
data.to_parquet(save_file_path)

print(f"--- Đã lưu file thành công vào: {save_file_path} ---")

In [ ]:
# Loading the dataset
data = pd.read_parquet('/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Cleaned_Base.parquet')
print("Dữ liệu sẵn sàng")

###Mã hóa Nhãn

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("--- Phân bố các loại Nhãn trong bộ dữ liệu ---")

# 1. Tính toán số lượng và tỷ lệ %
label_counts = data['Label'].value_counts()
label_percents = data['Label'].value_counts(normalize=True) * 100

# 2. Tạo một bảng đẹp để xem
dist_df = pd.DataFrame({
    'Số lượng (Count)': label_counts,
    'Tỷ lệ (%)': label_percents
})

# In bảng ra
print(dist_df)

# 3. Vẽ biểu đồ để thấy sự mất cân bằng KHỦNG KHIẾP
plt.figure(figsize=(12, 6))
# Dùng thang log (yscale='log') vì BENIGN quá lớn so với các loại khác
sns.barplot(x=label_counts.index, y=label_counts.values)
plt.yscale('log')
plt.xticks(rotation=90) # Xoay tên nhãn dọc xuống cho dễ đọc
plt.title('Phân bố các loại tấn công (Thang đo Log)')
plt.ylabel('Số lượng (Log Scale)')
plt.xlabel('Loại Tấn công')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np
import json

print("--- BẮT ĐẦU QUÁ TRÌNH MÃ HÓA & LƯU TRỮ ---")

# 1. SỬA LỖI HIỂN THỊ
data['Label'] = data['Label'].astype(str).str.replace('\ufffd', '-', regex=False)

# 2. Khởi tạo LabelEncoder
le = LabelEncoder()

# 3. Thực hiện Mã hóa
print("Đang mã hóa cột 'Label' từ Chữ sang Số...")
data['Label'] = le.fit_transform(data['Label'])

# 4. Tạo "Sổ tay Mã hóa" (Mapping)
mapping = dict(zip(le.classes_, le.transform(le.classes_)))

# Chuyển key thành string để in
mapping_str = {str(k): int(v) for k, v in mapping.items()}

print("\n--- SỔ TAY MÃ HÓA ---")
print(json.dumps(mapping_str, indent=4, ensure_ascii=False))

# 5. Lưu ra file Parquet
save_file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded.parquet"
print(f"\nĐang lưu file đã mã hóa vào: {save_file_path}...")
data.to_parquet(save_file_path)

print("--- HOÀN TẤT! ---")

In [ ]:
data.head(5)

###Xóa các cột chết

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Kết nối Drive & Load dữ liệu
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded.parquet"
df = pd.read_parquet(file_path)

print(f"🔹 Tổng số cột ban đầu: {df.shape[1]}")

# --- PHÂN TÍCH & XỬ LÝ: CÁC CỘT "CHẾT" (ZERO VARIANCE) ---
print("\n--- 1. PHÂN TÍCH CÁC CỘT 'CHẾT' ---")

# Tính độ lệch chuẩn
std = df.std(numeric_only=True)
dead_cols = std[std == 0].index.tolist()

# TẠO BẢNG CHỨNG MINH (EVIDENCE TABLE)
# Lấy các chỉ số thống kê cơ bản: Count, Mean, Std, Min, Max
evidence_table = df[dead_cols].describe().T[['count', 'mean', 'std', 'min', 'max']]

# Thêm cột 'Unique Values' để khẳng định chỉ có 1 giá trị duy nhất
evidence_table['n_unique'] = df[dead_cols].nunique()

print(f"-> Phát hiện {len(dead_cols)} đặc trưng có phương sai bằng 0.")
print("BẢNG 3.1: THỐNG KÊ CÁC ĐẶC TRƯNG BỊ LOẠI BỎ")
print(evidence_table)

# --- THỰC HIỆN XÓA ---
print("\n--- 2. TIẾN HÀNH LOẠI BỎ ---")
df_clean = df.drop(columns=dead_cols)

print(f"✅ Đã xóa {len(dead_cols)} cột.")
print(f"🔹 Tổng số cột sau khi xóa: {df_clean.shape[1]}")

# (Tùy chọn) Lưu đè lên biến df để dùng cho bước sau
df = df_clean

In [ ]:
# Đường dẫn file mới
save_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded_NoDeadCols.parquet"

print(f"Đang lưu dữ liệu vào: {save_path}...")
df.to_parquet(save_path)

print("✅ Đã lưu file thành công!")

##Trực quan hóa dữ liệu

###Vẽ Bản đồ Nhiệt (Heatmap) Tương quan.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Kết nối Drive & Load TOÀN BỘ dữ liệu
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded_NoDeadCols.parquet"

print(f"Đang đọc toàn bộ file: {file_path}...")
df = pd.read_parquet(file_path)
print(f"Kích thước dữ liệu: {df.shape}")

# 2. Xóa cột Label trước khi tính (để chỉ tính tương quan giữa các đặc trưng)
if 'Label' in df.columns:
    df_input = df.drop(columns=['Label'])
else:
    df_input = df

# 3. Tính toán Ma trận tương quan trên TOÀN BỘ dữ liệu
print("Đang tính toán ma trận tương quan (Full Dataset)... Vui lòng đợi xíu...")
# numeric_only=True để đảm bảo chỉ tính số, tránh lỗi nếu còn sót cột string
corr = df_input.corr(numeric_only=True).round(2)

# 4. Hiển thị bảng màu
print("Đang hiển thị bảng tương quan...")
corr.style.background_gradient(cmap='coolwarm', axis=None).format(precision=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Giả sử biến 'corr' đã được tính ở bước trước
print("Đang vẽ biểu đồ Heatmap (Size 24x24)...")

fig, ax = plt.subplots(figsize = (24, 24))

# annot=False: Tắt số liệu để đỡ rối
# linewidth=0.5: Kẻ viền trắng cho dễ nhìn
sns.heatmap(corr, cmap = 'coolwarm', annot = False, linewidth = 0.5)

plt.title('Correlation Matrix (Ma trận Tương quan)', fontsize = 18)
plt.show()

Xóa các cột trùng lặp (High Correlation > 0.95) - Toàn bộ dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Load dữ liệu
# drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded_NoDeadCols.parquet"
df = pd.read_parquet(file_path)

print(f"🔹 Kích thước ban đầu: {df.shape}")

# 2. Tách biến đầu vào để tính toán (Bỏ cột Label)
X = df.drop(columns=['Label'])

# 3. Tính Ma trận tương quan trên TOÀN BỘ dữ liệu
print("⏳ Đang tính ma trận tương quan trên toàn bộ dữ liệu (Có thể mất vài phút)...")
# Lưu ý: Việc này tốn RAM
corr_matrix = X.corr(numeric_only=True).abs()

# 4. Tìm các cặp cột trùng lặp (> 0.95)
# Chỉ lấy tam giác trên của ma trận
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Lập danh sách các cột cần xóa
to_drop_high_corr = [column for column in upper.columns if any(upper[column] > 0.95)]

print(f"   -> Phát hiện {len(to_drop_high_corr)} cột bị trùng lặp thông tin.")
print(f"   -> Danh sách: {to_drop_high_corr}")

# 5. Thực hiện xóa trên dữ liệu gốc
df = df.drop(columns=to_drop_high_corr)

# Giải phóng bộ nhớ RAM
del corr_matrix, upper, X
import gc
gc.collect()

print(f"✅ Đã xóa xong bước 1! Kích thước hiện tại: {df.shape}")

VẼ BIỂU ĐỒ TƯƠNG QUAN VỚI NHÃN (LABEL) TRÊN DỮ LIỆU ĐÃ LÀM SẠCH

In [ ]:
# ---------------------------------------------------------
# VẼ BIỂU ĐỒ TƯƠNG QUAN VỚI NHÃN (LABEL) TRÊN DỮ LIỆU ĐÃ LÀM SẠCH
# ---------------------------------------------------------

# Tính tương quan với Label
print("⏳ Đang tính toán mức độ ảnh hưởng đến Label...")

# Tách X (Features) và y (Label) từ mẫu mới
X_viz = df_viz_sample.drop(columns=['Label'])
y_viz = df_viz_sample['Label']

# Tính tương quan (Lấy trị tuyệt đối)
corr_with_target = X_viz.corrwith(y_viz).abs()

# Sắp xếp từ cao xuống thấp
feature_importance = corr_with_target.sort_values(ascending=False)

plt.figure(figsize=(10, 8)) # Điều chỉnh kích thước cho gọn hơn vì ít cột hơn
top_10 = feature_importance.head(10) # Lấy 10 thay vì 20

# Vẽ biểu đồ
sns.barplot(x=top_10.values, y=top_10.index, palette='viridis', hue=top_10.index, legend=False)

plt.title('Top 10 Đặc trưng ảnh hưởng nhất', fontsize=15)
plt.xlabel('Hệ số tương quan tuyệt đối (|Correlation|)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

# In ra danh sách các cột "Vô dụng" (Tương quan < 0.01)
low_corr_cols = feature_importance[feature_importance < 0.01].index.tolist()
print("-" * 50)
print(f"🔍 Có {len(low_corr_cols)} đặc trưng còn lại có tương quan rất thấp với Nhãn (< 0.01):")
print(low_corr_cols)

Xóa các cột ít quan trọng (Tương quan với Label < 0.01) - Toàn bộ dữ liệu

In [ ]:
# (Tiếp tục sử dụng biến 'df' từ ô trên)

# 1. Tính tương quan giữa từng cột với Nhãn trên TOÀN BỘ dữ liệu
print("⏳ Đang tính độ tương quan với Nhãn trên toàn bộ dữ liệu...")

X_full = df.drop(columns=['Label'])
y_full = df['Label']

# Tính toán
corr_with_target = X_full.corrwith(y_full).abs()

# 2. Tìm các cột có tương quan quá thấp (< 0.01)
to_drop_low_corr = corr_with_target[corr_with_target < 0.01].index.tolist()

print(f"   -> Phát hiện {len(to_drop_low_corr)} cột có tương quan quá thấp (< 0.01).")
print(f"   -> Danh sách: {to_drop_low_corr}")

# 3. Thực hiện xóa
df_final = df.drop(columns=to_drop_low_corr)

print("-" * 40)
print(f"🎉 HOÀN TẤT LÀM SẠCH TRÊN TOÀN BỘ DỮ LIỆU!")
print(f"🔹 Kích thước cuối cùng: {df_final.shape}")

# 4. Lưu file sạch
save_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Clean_Features.parquet"
df_final.to_parquet(save_path)
print(f"💾 Đã lưu file chuẩn vào: {save_path}")

Vẽ biểu đồ    để  

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from google.colab import drive

# 1. Kết nối Drive & Load dữ liệu
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Encoded_NoDeadCols.parquet"
df = pd.read_parquet(file_path)

# 2. THỰC HIỆN LẤY MẪU PHÂN TẦNG (STRATIFIED SAMPLING)
# Chúng ta muốn lấy 50,000 mẫu
# stratify=df['Label']: Lệnh quan trọng để giữ nguyên tỷ lệ các loại tấn công
print("Đang thực hiện lấy mẫu phân tầng (giữ nguyên tỷ lệ các lớp)...")

try:
    df_stratified, _ = train_test_split(
        df,
        train_size=50000,      # Lấy 50k dòng
        stratify=df['Label'],
        random_state=42
    )
    print(f"Lấy mẫu thành công! Kích thước mẫu: {df_stratified.shape}")
except ValueError:
    # Phòng trường hợp có lớp quá ít (như Heartbleed) khiến hàm lỗi
    print("Cảnh báo: Một số lớp quá ít mẫu để phân tầng chuẩn, chuyển sang lấy mẫu ngẫu nhiên...")
    df_stratified = df.sample(n=50000, random_state=42)

# Sắp xếp lại Label theo tên để vẽ cho đẹp
df_stratified = df_stratified.sort_values('Label')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. NHẬP SỔ TAY MÃ HÓA (User cung cấp)
# Đây là bảng tra cứu ngược: Tên -> Số
encoding_dict = {
    "BENIGN": 0,
    "Bot": 1,
    "DDoS": 2,
    "DoS GoldenEye": 3,
    "DoS Hulk": 4,
    "DoS Slowhttptest": 5,
    "DoS slowloris": 6,
    "FTP-Patator": 7,
    "Heartbleed": 8,
    "Infiltration": 9,
    "PortScan": 10,
    "SSH-Patator": 11,
    "Web Attack - Brute Force": 12,
    "Web Attack - Sql Injection": 13,
    "Web Attack - XSS": 14
}

# 2. TẠO TỪ ĐIỂN GIẢI MÃ (Đảo ngược lại: Số -> Tên)
# Kết quả sẽ là: {0: 'BENIGN', 1: 'Bot', ...}
decoding_dict = {v: k for k, v in encoding_dict.items()}

# 3. ÁP DỤNG VÀO DỮ LIỆU
if 'df_stratified' in locals():
    # Tạo cột tên mới
    df_stratified['Label_Name'] = df_stratified['Label'].map(decoding_dict)

    # Sắp xếp thứ tự vẽ: BENIGN đầu tiên, các cái khác theo sau
    # Lấy danh sách các nhãn thực tế có trong mẫu
    present_labels = df_stratified['Label_Name'].unique().tolist()
    # Đưa BENIGN lên đầu (nếu có)
    if 'BENIGN' in present_labels:
        present_labels.remove('BENIGN')
        sort_order = ['BENIGN'] + sorted(present_labels)
    else:
        sort_order = sorted(present_labels)

    print("✅ Đã giải mã thành công! Cột 'Label_Name' đã sẵn sàng để vẽ.")
    print(f"Các loại tấn công có trong mẫu: {sort_order}")
else:
    print("⚠️ Lỗi: Chưa tìm thấy biến 'df_stratified'.")

In [ ]:
plt.figure(figsize=(16, 8))

# Vẽ Boxplot với trục X là tên đã giải mã
sns.boxplot(
    x='Label_Name',
    y='Flow Bytes/s',
    data=df_stratified,
    palette='viridis',
    order=sort_order # Vẽ theo thứ tự đã sắp xếp
)

# Cấu hình
plt.yscale('log') # Thang đo Logarit bắt buộc
plt.title('Phân bố FLOW BYTES/S (Tốc độ truyền tải) theo từng loại tấn công', fontsize=16)
plt.ylabel('Flow Bytes/s (Log Scale)', fontsize=12)
plt.xlabel('Loại tấn công', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11) # Xoay tên cho dễ đọc
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(16, 8))

sns.boxplot(
    x='Label_Name',
    y='Total Fwd Packets',
    data=df_stratified,
    palette='magma',
    order=sort_order
)

plt.yscale('log')
plt.title('Phân bố TOTAL FWD PACKETS (Số lượng gói tin gửi đi)', fontsize=16)
plt.ylabel('Số gói tin (Log Scale)', fontsize=12)
plt.xlabel('Loại tấn công', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(16, 8))

sns.boxplot(
    x='Label_Name',
    y='Packet Length Mean',
    data=df_stratified,
    palette='coolwarm',
    order=sort_order
)

plt.yscale('log')
plt.title('Phân bố PACKET LENGTH MEAN (Độ dài trung bình gói tin)', fontsize=16)
plt.ylabel('Độ dài (Bytes) - Log Scale', fontsize=12)
plt.xlabel('Loại tấn công', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(16, 8))

# Vẽ Boxplot
sns.boxplot(
    x='Label_Name',
    y='Flow IAT Mean',
    data=df_stratified,
    palette='rocket',
    order=sort_order
)

# Cấu hình quan trọng
plt.yscale('log') # BẮT BUỘC phải dùng Log vì chênh lệch thời gian cực lớn
plt.title('Phân bố FLOW IAT MEAN (Thời gian trung bình giữa các gói tin)', fontsize=16)
plt.ylabel('Thời gian (Microseconds) - Log Scale', fontsize=12)
plt.xlabel('Loại tấn công', fontsize=12)

# Xoay tên
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.3)

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# 1. Load dữ liệu sạch
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Clean_Features.parquet"
df = pd.read_parquet(file_path)

# 2. Tạo từ điển giải mã (Số -> Tên) để hiển thị cho đẹp
label_mapping = {
    0: 'BENIGN',
    1: 'Bot',
    2: 'DDoS',
    3: 'DoS GoldenEye',
    4: 'DoS Hulk',
    5: 'DoS Slowhttptest',
    6: 'DoS slowloris',
    7: 'FTP-Patator',
    8: 'Heartbleed',
    9: 'Infiltration',
    10: 'PortScan',
    11: 'SSH-Patator',
    12: 'Web Attack - Brute Force',
    13: 'Web Attack - Sql Injection',
    14: 'Web Attack - XSS'
}

# Tạo cột tên nhãn tạm thời để vẽ
df['Label_Name'] = df['Label'].map(label_mapping)

# 3. Tính toán thống kê
label_counts = df['Label_Name'].value_counts()
print("📊 Thống kê số lượng từng lớp:")
print(label_counts)

# 4. Vẽ biểu đồ
plt.figure(figsize=(14, 8))

# Dùng countplot của Seaborn
ax = sns.countplot(
    x='Label_Name',
    data=df,
    order=label_counts.index, # Sắp xếp từ cao xuống thấp
    palette='viridis'
)

# Cấu hình thang đo Logarit (QUAN TRỌNG: Vì dữ liệu rất mất cân bằng)
plt.yscale('log')

plt.title('Phân bố số lượng các lớp tấn công', fontsize=16)
plt.xlabel('Loại tấn công', fontsize=12)
plt.ylabel('Số lượng mẫu (Log Scale)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11) # Xoay tên cho dễ đọc
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 5. Hiển thị con số cụ thể trên đầu mỗi cột
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha = 'center', va = 'center',
                xytext = (0, 10),
                textcoords = 'offset points',
                fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dữ liệu (nếu chưa có)
# file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Clean_Features.parquet"
# df = pd.read_parquet(file_path)

# Tạo lại từ điển tên cho dễ đọc
label_mapping = {
    0: 'BENIGN', 1: 'Bot', 2: 'DDoS', 3: 'DoS GoldenEye', 4: 'DoS Hulk',
    5: 'DoS Slowhttptest', 6: 'DoS slowloris', 7: 'FTP-Patator', 8: 'Heartbleed',
    9: 'Infiltration', 10: 'PortScan', 11: 'SSH-Patator',
    12: 'Web Attack - Brute Force', 13: 'Web Attack - Sql Injection', 14: 'Web Attack - XSS'
}
df['Label_Name'] = df['Label'].map(label_mapping)

# 2. TÍNH TOÁN TỶ LỆ PHẦN TRĂM
total_count = len(df)
stats_df = df['Label_Name'].value_counts().to_frame(name='Count')
stats_df['Percentage'] = (stats_df['Count'] / total_count) * 100

print("📊 BẢNG THỐNG KÊ CHI TIẾT:")
print(stats_df)

# 3. VẼ BIỂU ĐỒ
plt.figure(figsize=(14, 8))

# Vẽ biểu đồ cột
ax = sns.barplot(
    x=stats_df.index,
    y=stats_df['Count'], # Vẫn dùng Count cho trục Y để dùng Log scale
    palette='viridis'
)

# Cấu hình thang đo Logarit (BẮT BUỘC để nhìn thấy các lớp nhỏ)
plt.yscale('log')

plt.title('Phân bố các lớp tấn công (Tỷ lệ %)', fontsize=16)
plt.xlabel('Loại tấn công', fontsize=12)
plt.ylabel('Số lượng mẫu (Log Scale)', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 4. HIỂN THỊ % TRÊN ĐẦU CỘT
for p in ax.patches:
    height = p.get_height()
    if height > 0: # Chỉ ghi chú nếu có dữ liệu
        percentage = (height / total_count) * 100
        # Định dạng: Ví dụ "83.1%" hoặc "0.01%"
        label_text = f'{percentage:.2f}%' if percentage >= 0.01 else '<0.01%'

        ax.annotate(label_text,
                    (p.get_x() + p.get_width() / 2., height),
                    ha = 'center', va = 'bottom',
                    xytext = (0, 5),
                    textcoords = 'offset points',
                    fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

####Phân chia tệp dữ liệu và chuẩn hóa

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive

# 1. Load dữ liệu sạch
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/DoAn_NIDS/Dataset/CICIDS2017_Clean_Features.parquet"
df = pd.read_parquet(file_path)

print(f"🔹 Kích thước dữ liệu gốc: {df.shape}")

# 2. Phân chia Train/Test
# Tách X, y
X = df.drop(columns=['Label'])
y = df['Label']

print("⏳ Đang chia tập dữ liệu (80/20)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. CHUẨN HÓA DỮ LIỆU (Min-Max Scaling)
print("⏳ Đang thực hiện chuẩn hóa (Min-Max Scaling)...")

# Khởi tạo bộ scaler
scaler = MinMaxScaler()

# LƯU Ý QUAN TRỌNG:
# Chỉ fit (học tham số min, max) trên tập TRAIN
X_train_scaled = scaler.fit_transform(X_train)

# Sau đó transform (áp dụng) lên tập TEST
# (Tuyệt đối không fit trên Test để tránh lộ dữ liệu)
X_test_scaled = scaler.transform(X_test)

# 4. Kiểm tra kết quả
print("-" * 40)
print("✅ CHUẨN HÓA HOÀN TẤT!")
print(f"   - Min của Train sau scale: {X_train_scaled.min()}")
print(f"   - Max của Train sau scale: {X_train_scaled.max()}")
print(f"   - Kích thước Train: {X_train_scaled.shape}")
print(f"   - Kích thước Test:  {X_test_scaled.shape}")
print("-" * 40)

In [ ]:
X_train_scaled[:1]

In [ ]:
import pandas as pd

print("-" * 50)
print("📊 THỐNG KÊ CHI TIẾT SỐ LƯỢNG NHÃN (SAU KHI CHIA TRAIN/TEST)")
print("-" * 50)

# 1. Thống kê tập TRAIN
print(f"1️⃣ TẬP HUẤN LUYỆN (TRAIN SET) - Tổng: {len(y_train)} mẫu")
train_counts = y_train.value_counts()
print(train_counts)

# 2. Thống kê tập TEST
print(f"\n2️⃣ TẬP KIỂM THỬ (TEST SET) - Tổng: {len(y_test)} mẫu")
test_counts = y_test.value_counts()
print(test_counts)

# 3. Kiểm tra nhanh tỷ lệ phân chia (Stratify)
print("-" * 50)
print("🔍 KIỂM TRA TỶ LỆ MỘT LỚP BẤT KỲ (Ví dụ: BENIGN)")
benign_train_pct = (train_counts.get(0, 0) / len(y_train)) * 100
benign_test_pct = (test_counts.get(0, 0) / len(y_test)) * 100

print(f"   - Tỷ lệ Benign trong Train: {benign_train_pct:.2f}%")
print(f"   - Tỷ lệ Benign trong Test:  {benign_test_pct:.2f}%")
print("   -> (Nếu hai số này xấp xỉ nhau nghĩa là tham số stratify=y đã hoạt động tốt)")

Chuẩn bị dữ liệu cho bài toán nhị phân

In [ ]:
import os
import joblib
import numpy as np
from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from sklearn.utils import class_weight, shuffle
from google.colab import drive

# 1. Kết nối Drive & Tạo thư mục lưu trữ
try:
    drive.mount('/content/drive')
except:
    pass

BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
PATH_BINARY = os.path.join(BASE_DIR, "Binary_Data/") # Thư mục riêng cho Binary

# Tạo thư mục nếu chưa có
os.makedirs(PATH_BINARY, exist_ok=True)

# Cấu hình số lượng mục tiêu
TARGET_BENIGN = 500000   # Giảm Benign xuống 500k

print("-" * 50)
print(f"🚀 BẮT ĐẦU XỬ LÝ DỮ LIỆU NHỊ PHÂN")
print(f"📂 Lưu tại: {PATH_BINARY}")
print("-" * 50)

In [ ]:
# ==============================================================================
# BƯỚC 1: GIẢM DỮ LIỆU (UNDERSAMPLING)
# ==============================================================================
# Chiến thuật: Chỉ giảm lớp 0 (Benign) xuống 500k, các lớp Tấn công giữ nguyên
print(f"1️⃣ Đang giảm lớp Benign xuống {TARGET_BENIGN} mẫu...")

under_strategy = {0: TARGET_BENIGN}
rus = RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)

# X_train_scaled và y_train là biến có từ bước chuẩn hóa trước đó
X_train_bin, y_train_bin = rus.fit_resample(X_train_scaled, y_train)

print(f"   -> Kích thước sau khi giảm: {X_train_bin.shape}")

In [ ]:
# ==============================================================================
# BƯỚC 2: MÃ HÓA NHÃN (LABEL ENCODING)
# ==============================================================================
# Quy tắc: 0 -> 0 (Bình thường), >0 -> 1 (Tấn công)
print("2️⃣ Đang mã hóa nhãn về 0 và 1...")

y_train_bin = np.where(y_train_bin == 0, 0, 1)
y_test_bin = np.where(y_test == 0, 0, 1) # Áp dụng tương tự cho tập Test

# ==============================================================================
# BƯỚC 3: TRỘN ĐỀU (SHUFFLE)
# ==============================================================================
# Để tránh dữ liệu bị sắp xếp theo thứ tự lớp làm Model học lệch
print("3️⃣ Đang trộn đều dữ liệu (Shuffle)...")
X_train_bin, y_train_bin = shuffle(X_train_bin, y_train_bin, random_state=42)

# ==============================================================================
# BƯỚC 4: TÍNH TRỌNG SỐ (CLASS WEIGHTS)
# ==============================================================================
# Dù đã giảm Benign, dữ liệu vẫn có thể chênh lệch. Cần tính weights để cân bằng khi Train.
print("4️⃣ Đang tính toán Class Weights...")

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_bin),
    y=y_train_bin
)
class_weights_dict = {i: weights[i] for i in range(len(weights))}
print(f"   -> Trọng số: {class_weights_dict}")

# ==============================================================================
# BƯỚC 5: LƯU TRỮ (SAVING)
# ==============================================================================
print("5️⃣ Đang lưu toàn bộ vào thư mục Binary_Data...")

# Lưu tập Train
joblib.dump(X_train_bin, PATH_BINARY + 'X_train.pkl')
joblib.dump(y_train_bin, PATH_BINARY + 'y_train.pkl')

# Lưu tập Test (X_test gốc đã scale, y_test đã đổi nhãn 0/1)
joblib.dump(X_test_scaled, PATH_BINARY + 'X_test.pkl')
joblib.dump(y_test_bin, PATH_BINARY + 'y_test.pkl')

# Lưu trọng số
joblib.dump(class_weights_dict, PATH_BINARY + 'class_weights.pkl')

print("\n✅ HOÀN TẤT! SỐ LIỆU CUỐI CÙNG:")
print(f"   - Train Normal (0): {Counter(y_train_bin)[0]}")
print(f"   - Train Attack (1): {Counter(y_train_bin)[1]}")
print(f"   - Test Normal  (0): {Counter(y_test_bin)[0]}")
print(f"   - Test Attack  (1): {Counter(y_test_bin)[1]}")

In [ ]:
import joblib

# Load file trọng số vừa lưu
save_dir = "/content/drive/MyDrive/DoAn_NIDS/Dataset/Binary_Data/"
weights_dict = joblib.load(save_dir + 'class_weights.pkl')

print("-" * 40)
print("⚖️ TRỌNG SỐ THỰC TẾ (CLASS WEIGHTS):")
print(f"   - Class 0 (Benign): {weights_dict[0]:.4f}")
print(f"   - Class 1 (Attack): {weights_dict[1]:.4f}")

# Giải thích ý nghĩa
if abs(weights_dict[0] - weights_dict[1]) < 0.2:
    print("\n✅ NHẬN XÉT: Trọng số xấp xỉ nhau (gần 1).")
    print("   -> Dữ liệu rất cân bằng, mô hình sẽ học cực kỳ ổn định!")
else:
    print("\n⚠️ NHẬN XÉT: Có sự chênh lệch nhẹ, việc dùng trọng số là cần thiết.")
print("-" * 40)

In [ ]:
# ... (Phần import thư viện giữ nguyên) ...

print("🚀 BẮT ĐẦU QUY TRÌNH XỬ LÝ DỮ LIỆU ĐA LỚP (MULTICLASS)...")

# ==============================================================================
# ⚠️ CẬP NHẬT QUAN TRỌNG: SỬ DỤNG DỮ LIỆU ĐÃ CHUẨN HÓA
# ==============================================================================
# Biến X_train và X_test gốc (DataFrame) chưa được scale.
# Ta phải thay thế chúng bằng bản đã scale từ bước trước (Numpy Array).

print("🔄 Đang cập nhật biến X sang dạng đã chuẩn hóa (Min-Max)...")

# Cập nhật cho quy trình xử lý
X_train_multi_input = X_train_scaled  # Dữ liệu Train đã scale
X_test_multi_input = X_test_scaled    # Dữ liệu Test đã scale

# Kiểm tra nhanh xem có bị NaN không (đề phòng)
if np.isnan(X_train_multi_input).any():
    print("⚠️ Cảnh báo: Dữ liệu sau chuẩn hóa có chứa NaN. Đang xử lý...")
    X_train_multi_input = np.nan_to_num(X_train_multi_input)
    X_test_multi_input = np.nan_to_num(X_test_multi_input)

print(f"✅ Đã cập nhật X_train đầu vào: {X_train_multi_input.shape}")

# ==============================================================================
# TIẾP TỤC QUY TRÌNH MAPPING (Như cũ)
# ==============================================================================

# --- 1. ĐỊNH NGHĨA MAPPING ---
# ... (Giữ nguyên phần dictionary mapping_strategy của anh) ...

###Chuẩn bị dữ liệu cho bài toán đa lớp

Bước 1: Gộp nhãn

In [ ]:
import pandas as pd
import numpy as np
import joblib
from collections import Counter
from sklearn.utils import shuffle, class_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import os # Import os module if not already present

# Define base directory for saving data in Google Drive
BASE_DIR = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"
PATH_MULTI = os.path.join(BASE_DIR, "Multi_Data/") # Change to Drive path

# Create directory if it doesn't exist
if not os.path.exists(PATH_MULTI):
    os.makedirs(PATH_MULTI)

print("🚀 BẮT ĐẦU QUY TRÌNH XỬ LÝ DỮ LIỆU ĐA LỚP (MULTICLASS)...")

# --- 1. ĐỊNH NGHĨA MAPPING (Dựa trên sổ tay) ---
# 0: Benign
# 1: DoS/DDoS
# 2: PortScan
# 3: BruteForce
# 4: Other/Rare

mapping_strategy = {
    # --- Benign ---
    0: 0,

    # --- DoS/DDoS ---
    2: 1,   # DDoS
    3: 1,   # DoS GoldenEye
    4: 1,   # DoS Hulk
    5: 1,   # DoS Slowhttptest
    6: 1,   # DoS slowloris

    # --- PortScan ---
    10: 2,  # PortScan

    # --- BruteForce ---
    7: 3,   # FTP-Patator
    11: 3,  # SSH-Patator
    12: 3,  # Web Attack - Brute Force (Xếp vào đây hợp lý hơn Web)

    # --- Other / Rare (Các loại hiếm hoặc rời rạc) ---
    1: 4,   # Bot
    8: 4,   # Heartbleed (Rất hiếm)
    9: 4,   # Infiltration
    13: 4,  # Web Attack - Sql Injection
    14: 4   # Web Attack - XSS
}

print("🔄 ĐANG THỰC HIỆN GỘP NHÃN THEO CHIẾN LƯỢC MỚI...")

# --- 2. ÁP DỤNG CHO DỮ LIỆU ---
# .map() sẽ thay thế giá trị cũ bằng giá trị mới theo từ điển trên
# Áp dụng mapping
y_train_multi = y_train.map(mapping_strategy)
y_test_multi = y_test.map(mapping_strategy)

# --- 3. KIỂM TRA KẾT QUẢ ---
print("-" * 50)
print("📊 SỐ LƯỢNG MẪU TỪNG NHÓM SAU KHI GỘP (TRAIN)")
print("-" * 50)
print(y_train_multi.value_counts().sort_index())

print("\n🏷️ CHÚ THÍCH NHÃN MỚI:")
print("0: Benign")
print("1: DoS/DDoS")
print("2: PortScan")
print("3: BruteForce")
print("4: Other/Rare")

Chiến lược cân bằng:
 + Giảm lớp Benign xuống 300k
 + Giữ nguyên lớp 1, 2
 + SMOTE 2 lớp 3,4 lên 30k


In [ ]:
# ==============================================================================
# BƯỚC 2: CÂN BẰNG DỮ LIỆU (UNDERSAMPLE & SMOTE)
# ==============================================================================
print("\n2️⃣ Đang thực hiện Cân bằng dữ liệu (Custom Strategy)...")

# Chiến lược: Benign -> 300k | DoS, PortScan -> Giữ nguyên | BruteForce, Other -> 30k
under_strategy = {0: 300000}
over_strategy = {3: 30000, 4: 30000}

# Pipeline
pipeline = Pipeline(steps=[
    ('u', RandomUnderSampler(sampling_strategy=under_strategy, random_state=42)),
    ('s', SMOTE(sampling_strategy=over_strategy, random_state=42, k_neighbors=3))
])

X_train_multi, y_train_multi = pipeline.fit_resample(X_train, y_train_multi)
print(f"   -> Phân phối sau cân bằng: {Counter(y_train_multi)}")

In [ ]:
# ==============================================================================
# BƯỚC 3: TRỘN ĐỀU (SHUFFLE)
# ==============================================================================
print("\n3️⃣ Đang trộn đều dữ liệu (Shuffle)...")
X_train_multi, y_train_multi = shuffle(X_train_multi, y_train_multi, random_state=42)

# ==============================================================================
# BƯỚC 4: TÍNH TRỌNG SỐ (CLASS WEIGHTS)
# ==============================================================================
print("\n4️⃣ Đang tính toán Class Weights...")

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_multi),
    y=y_train_multi
)
class_weights_dict = {i: weights[i] for i in range(len(weights))}
print(f"   -> Trọng số 5 lớp: {class_weights_dict}")


In [ ]:
from tensorflow.keras.utils import to_categorical

# ==============================================================================
# BƯỚC 4.5: ONE-HOT ENCODING (MỚI THÊM)
# ==============================================================================
print("\n⚡ Đang thực hiện One-Hot Encoding cho Deep Learning...")

# Chuyển đổi nhãn từ số nguyên (0,1,2..) sang vector (VD: [0,0,1,0,0])
# num_classes=5 tương ứng với 5 nhóm mình đã gộp
y_train_onehot = to_categorical(y_train_multi, num_classes=5)
y_test_onehot = to_categorical(y_test_multi, num_classes=5)

print(f"   -> Kích thước y_train sau One-Hot: {y_train_onehot.shape}")
print(f"   -> Kích thước y_test sau One-Hot:  {y_test_onehot.shape}")

In [ ]:
# ==============================================================================
# BƯỚC 5: LƯU TRỮ (SAVING)
# ==============================================================================
print("\n5️⃣ Đang lưu toàn bộ vào thư mục Multi_Data...")

# Lưu tập Train
joblib.dump(X_train_multi, PATH_MULTI + 'X_train_multi.pkl')
joblib.dump(y_train_onehot, PATH_MULTI + 'y_train_multi.pkl') # Lưu bản One-Hot

# Lưu tập Test
joblib.dump(X_test, PATH_MULTI + 'X_test_multi.pkl')
joblib.dump(y_test_onehot, PATH_MULTI + 'y_test_multi.pkl')   # Lưu bản One-Hot

# Lưu trọng số (để dùng khi fit model)
joblib.dump(class_weights_dict, PATH_MULTI + 'class_weights_multi.pkl')

print("-" * 50)
print("✅ HOÀN TẤT QUY TRÌNH MULTICLASS (ĐÃ ONE-HOT)!")
print(f"   - Train X: {X_train_multi.shape} | y: {y_train_onehot.shape}")
print(f"   - Test  X: {X_test.shape}       | y: {y_test_onehot.shape}")
print("-" * 50)